In [2]:
# 変化量の上位95%だけを抽出
import pandas as pd
import os

# 入力ファイル
input_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/vad_dis.csv"

# 出力ファイル
output_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/Δv-95.csv"

# 出力先が無ければ新規作成
os.makedirs(os.path.dirname(output_csv), exist_ok=True)

# CSV読み込み
df = pd.read_csv(input_csv)

filter_95 = df[df["delta_valence"] > 0.503]

result = filter_95[
    [
        "filename",
        "text",
        "valence",
        "arousal",
        "dominance",
        "delta_valence",
        "delta_arousal",
        "delta_dominance",
    ]
]

# 保存
result.to_csv(output_csv, index=False, encoding="utf-8-sig")

print(f"保存完了: {output_csv}")
print(f"発話数: {len(result)}")

保存完了: /home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/Δv-95.csv
発話数: 13


In [3]:
df[df["delta_valence"] > 0.503]

,filename,text,valence,arousal,dominance,delta_valence,delta_arousal,delta_dominance
66,0142_R.wav,<FV>(F その),0.12,0.25,0.20,0.52,0.35,0.35
80,0163_R.wav,先行ってた,0.35,0.47,0.46,0.51,0.26,0.24
81,0164_R.wav,<笑>,0.89,0.75,0.72,0.54,0.28,0.26
103,0212_R.wav,<笑>,0.92,0.78,0.75,0.56,0.31,0.27
104,0215_R.wav,(F うん)(F うん),0.29,0.00,0.07,0.63,0.78,0.68
124,0259_R.wav,(F うーん),0.26,0.05,0.11,0.62,0.70,0.60
152,0319_R.wav,(F うーん),0.31,0.08,0.15,0.58,0.67,0.57
158,0331_R.wav,<笑>,0.86,0.71,0.69,0.55,0.62,0.59
184,0376_R.wav,<笑>,0.36,0.50,0.37,0.62,0.31,0.40
197,0401_R.wav,(F うーん),0.34,0.13,0.19,0.51,0.50,0.44


In [4]:
# パーセンタイル95%のRの発話
import pandas as pd
import os
import shutil

# ==========================
# パス
# ==========================
l_csv = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/vad_dis.csv"
wav_dir = "/home/mitani/CSJ-emo-int_bunseki/ALL-WAV"
output_root = "/home/mitani/CSJ-emo-int_bunseki/0718/R_bunseki/results/eval-v"

os.makedirs(output_root, exist_ok=True)

# ==========================
# CSV読込（Lのみ）
# ==========================
df_l = pd.read_csv(l_csv)

# 発話番号（CSVのインデックスではなく、0015_R.wav の「15」を指定）
change_idx = [
    142, 163, 164, 212, 215, 259, 319, 331, 376, 402, 404, 421
]

context = 3

# ==========================
# 抽出
# ==========================
for utt_no in change_idx:

    # 発話番号からファイル名を作成
    target_filename = f"{utt_no:04d}_R.wav"

    # filenameから該当行を検索
    matched = df_l[df_l["filename"] == target_filename]

    if matched.empty:
        print(f"{target_filename} がCSV内に見つかりません")
        continue

    # DataFrame内でのインデックス取得
    idx = matched.index[0]

    print(f"発話番号 {utt_no} -> DataFrame index {idx}")

    # 前後3発話（Rのみ）
    start = max(0, idx - context)
    end = min(len(df_l), idx + context + 1)

    context_df = df_l.iloc[start:end]

    # 保存フォルダ
    folder = os.path.join(
        output_root,
        os.path.splitext(target_filename)[0]
    )
    os.makedirs(folder, exist_ok=True)

    # context.csv保存
    context_df.to_csv(
        os.path.join(folder, "context.csv"),
        index=False,
        encoding="utf-8-sig"
    )

    # target情報保存
    with open(os.path.join(folder, "target.txt"), "w", encoding="utf-8") as f:
        f.write(f"Utterance number : {utt_no}\n")
        f.write(f"DataFrame index : {idx}\n")
        f.write(f"Target filename : {target_filename}\n")
        f.write(f"Context range : {start} - {end - 1}\n")

    # 音声コピー
    for _, row in context_df.iterrows():

        src = os.path.join(wav_dir, row["filename"])

        if not os.path.exists(src):
            print(f"Not found: {src}")
            continue

        # ターゲット音声にはTARGET_を付ける
        if row["filename"] == target_filename:
            dst_name = "TARGET_" + row["filename"]
        else:
            dst_name = row["filename"]

        dst = os.path.join(folder, dst_name)

        shutil.copy2(src, dst)

print("完了")

発話番号 142 -> DataFrame index 66
発話番号 163 -> DataFrame index 80
発話番号 164 -> DataFrame index 81
発話番号 212 -> DataFrame index 103
発話番号 215 -> DataFrame index 104
発話番号 259 -> DataFrame index 124
発話番号 319 -> DataFrame index 152
発話番号 331 -> DataFrame index 158
発話番号 376 -> DataFrame index 184
発話番号 402 -> DataFrame index 198
発話番号 404 -> DataFrame index 199
発話番号 421 -> DataFrame index 207
完了
